**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Software-Defined Radio

> ⚠️ **Draft — code not machine-verified.** Requires an RTL-SDR USB dongle (~$30) not available at authoring time. An instructor should run each block before teaching. Remove this banner after that pass.

The most convincing demo SPS owns: a $30 USB dongle turns the entire DSP track into **real signals from real antennas** — FM stations, aircraft transponders, weather satellites. Three sessions from unboxing to decoding.

## 1. Pre-requisites

- [Digital Communications](../Intro_DSP/Digital_Communications.ipynb) and [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) (multirate).
- Hardware: RTL-SDR v3/v4 dongle + telescopic antenna. Software: `pip install pyrtlsdr`, plus the `rtl-sdr` system drivers (Linux: `apt install rtl-sdr`; all platforms: see rtl-sdr.com quick-start).

---
### 🕐 Session 1 of 3 — *Hello, Spectrum* (~35 min)
**Goal:** capture live IQ samples; sweep the dial and read the local RF neighborhood.
**Feeds into:** Session 2 (FM receiver).

---

💡 **Intuition.** The dongle is a *mixer + sampler*: it shifts a chosen slice of the RF spectrum down to baseband and hands you **complex IQ samples** — the analytic signal the DSP track has been quietly preparing you for. From here, everything is software: the 'radio' in SDR is your NumPy code.

In [ ]:
from rtlsdr import RtlSdr
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig

sdr = RtlSdr()
sdr.sample_rate = 2.4e6            # 2.4 MHz of spectrum at once
sdr.center_freq = 100.3e6          # tune near the FM band (adjust to a local station!)
sdr.gain = "auto"

iq = sdr.read_samples(256 * 1024)  # complex64 IQ
sdr.close()
print(f"captured {len(iq):,} complex samples; mean power {10*np.log10(np.mean(np.abs(iq)**2)):.1f} dB")

In [ ]:
# your RF neighborhood: Welch PSD of the capture (Statistical SP, now on airwaves)
f, P = sig.welch(iq, fs=2.4e6, nperseg=4096, return_onesided=False)
f = np.fft.fftshift(f); P = np.fft.fftshift(P)
plt.figure(figsize=(9, 3))
plt.plot((f + 100.3e6) / 1e6, 10*np.log10(P))
plt.xlabel("frequency [MHz]"); plt.ylabel("PSD [dB/Hz]")
plt.title("live spectrum: each bump is a broadcaster — count your neighbors")
plt.tight_layout(); plt.show()

---
### 🕐 Session 2 of 3 — *A Working FM Receiver* (~40 min)
**Goal:** demodulate broadcast FM to audio in ~15 lines — every line a workshop you've taken.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (decoding digital).

---

💡 **Intuition.** FM encodes audio in the *rate of phase rotation* of the IQ samples. So the demodulator is one line: the angle between consecutive samples, `np.angle(iq[1:] * np.conj(iq[:-1]))`. Everything around that line is the [multirate](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) plumbing: decimate 2.4 MHz → 240 kHz (channel), demodulate, de-emphasize, decimate → 48 kHz (audio).

In [ ]:
# the 15-line FM receiver
chan = sig.decimate(iq, 10, ftype="fir")                       # 2.4 MHz → 240 kHz
demod = np.angle(chan[1:] * np.conj(chan[:-1]))                # THE line: instantaneous frequency
# de-emphasis: FM boosts treble at the transmitter; undo it (75 µs RC in the US)
b_de, a_de = [1 - np.exp(-1/(240e3*75e-6))], [1, -np.exp(-1/(240e3*75e-6))]
audio = sig.lfilter(b_de, a_de, demod)
audio = sig.decimate(audio, 5, ftype="fir")                    # 240 kHz → 48 kHz
audio /= np.abs(audio).max()

from scipy.io import wavfile
wavfile.write("fm_capture.wav", 48000, (audio * 32767).astype(np.int16))
print("wrote fm_capture.wav — play it. that's the radio, demodulated by your own code.")

---
### 🕐 Session 3 of 3 — *Decoding Digital: ADS-B* (~40 min)
**Goal:** receive aircraft transponders at 1090 MHz; detect real packets with a matched filter.
**Builds on:** Session 2; [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb) S4.

---

💡 **Intuition.** Aircraft broadcast position/identity as 1090 MHz pulse-position packets. The receiver is [detection theory](../Intro_DSP/Statistical_Signal_Processing.ipynb) verbatim: correlate against the known 8 µs preamble (matched filter), threshold (Neyman–Pearson), then slice bits by comparing pulse-position energies. Decoding the 56/112-bit payload (`pyModeS` does the field extraction) turns your antenna into a flight tracker.

In [ ]:
sdr = RtlSdr(); sdr.sample_rate = 2e6; sdr.center_freq = 1090e6; sdr.gain = 40
mag = np.abs(sdr.read_samples(4_000_000))          # 2 s of magnitude at 2 MS/s
sdr.close()

# ADS-B preamble at 2 MS/s: pulses at samples 0,2,7,9 in a 16-sample window
pre = np.zeros(16); pre[[0, 2, 7, 9]] = 1; pre -= pre.mean()
corr = np.correlate(mag - mag.mean(), pre, "valid")
th = corr.mean() + 6 * corr.std()                  # ~constant false-alarm threshold
hits = np.where(corr > th)[0]
# de-duplicate hits closer than one message length
msgs = hits[np.diff(hits, prepend=-999) > 240]
print(f"candidate ADS-B packets in 2 s: {len(msgs)}")
print("next step: slice bits from each hit and hand to pyModeS.decoder — see the pyModeS docs")

## 4. Conclusion

IQ samples in, software radio out: the spectrum browser is Welch, the FM receiver is one `angle()` plus multirate plumbing, and the aircraft tracker is a matched filter with a CFAR-style threshold. Total hardware bill: $30.

---
## Where next

- [Digital Communications](../Intro_DSP/Digital_Communications.ipynb) — now try *transmit-side* theory against received reality.
- [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) — make these pipelines run live instead of on captures.
- [Array Processing](../Intro_DSP/Array_Processing.ipynb) — two dongles, one shared clock: direction finding.